# Perhitungan DS Awal dan DS–PSO

Notebook ini menghitung ulang metrik dari data uji yang sama untuk DS awal dan 10 run PSO (seed 42–51). Fungsi inferensi, metrik, dan optimasi memakai `research/run_experiment.py`, yaitu program yang menghasilkan laporan eksperimen Bab IV. Dataset dipatok ke ekspor yang tercatat dalam `reports/experiment_report.json`; aturan awal dipatok ke `research/belief_bab_iv.json` sesuai Tabel 4.6 agar tabel historis dapat direproduksi. Jalankan semua sel dari atas ke bawah; optimasi 10 × 30 partikel × 100 iterasi memerlukan waktu.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from research.run_experiment import ExperimentEngine, constrain, file_hash, metrics, optimize

report_path = Path("reports/experiment_report.json")
arsip = json.loads(report_path.read_text(encoding="utf-8"))
metadata = arsip["metadata"]
train_path = Path(metadata["train"])
test_path = Path(metadata["test"])
belief_path = Path("research/belief_bab_iv.json")
for path in (train_path, test_path, belief_path):
    if not path.is_file():
        raise FileNotFoundError(path)
if file_hash(train_path) != metadata["train_sha256"] or file_hash(test_path) != metadata["test_sha256"]:
    raise ValueError("Dataset berbeda dari arsip Bab IV; pilih ekspor yang sesuai sebelum membandingkan angka.")

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
rules = json.loads(belief_path.read_text(encoding="utf-8"))["rules"]
classes = sorted(df_train["Kerusakan"].unique().tolist())
if set(df_test["Kerusakan"]) - set(classes):
    raise ValueError("Ada kelas uji yang tidak terdapat pada data latih.")
counts = df_train["Kerusakan"].value_counts()
priors = {name: float(counts.get(name, 0) / len(df_train)) for name in classes}
engine = ExperimentEngine(rules, classes, priors)
train_active = engine.active_rules(df_train)
test_active = engine.active_rules(df_test)
initial = constrain([hyp["initial_belief"] for rule in rules for hyp in rule["hypotheses"]], rules)

display(pd.DataFrame({"Berkas": [str(train_path), str(test_path)], "Jumlah kasus": [len(df_train), len(df_test)]}))
print(f"Aturan: {len(rules)}; parameter belief: {len(initial)}; kelas: {len(classes)}")

,Berkas,Jumlah kasus
0,data\dataLatih\dataLatih-2026-08-29-20-08.csv,736
1,data\dataUji\dataUji-2026-08-29-20-08.csv,185


Aturan: 17; parameter belief: 28; kelas: 14


## 1. DS awal pada data uji

Untuk tiap kasus, aturan aktif digabung dengan belief pakar lalu massa Θ dibagi ke seluruh kelas (probabilitas pignistik). Prediksi ialah kelas dengan probabilitas tertinggi. Accuracy = benar / jumlah uji; F1 macro merata-ratakan F1 setiap kelas; F1 weighted memakai support kelas. Multiclass Brier = rata-rata jumlah kuadrat `(probabilitas kelas - one-hot label aktual)^2` per kasus.

In [2]:
y_test = df_test["Kerusakan"].to_numpy()
baseline_prob = engine.matrix(test_active, initial)
baseline = metrics(y_test, baseline_prob, classes)
benar_ds = round(baseline["accuracy"] * len(y_test))
fmt_persen = lambda value: f"{value:.2%}".replace(".", ",")
print(f"Accuracy DS awal = {benar_ds}/{len(y_test)} x 100% = {fmt_persen(baseline['accuracy'])}")
display(pd.DataFrame({
    "Metrik": ["Accuracy", "Macro F1", "Weighted F1", "Brier score"],
    "DS awal": [fmt_persen(baseline["accuracy"]), fmt_persen(baseline["macro_f1"]),
                fmt_persen(baseline["weighted_f1"]), f"{baseline['multiclass_brier']:.4f}".replace(".", ",")],
}))

Accuracy DS awal = 154/185 x 100% = 83,24%


,Metrik,DS awal
0,Accuracy,"83,24%"
1,Macro F1,"61,94%"
2,Weighted F1,"83,10%"
3,Brier score,"0,3922"


## 2. Optimasi PSO dan evaluasi setiap seed

Fitness tiap partikel dihitung **hanya pada data latih** sebagai `1 - multiclass Brier`. Setiap seed menjalankan 30 partikel dan 100 iterasi. Setelah optimasi, belief terbaik seed tersebut dievaluasi pada data uji dengan fungsi metrik yang sama seperti DS awal.

In [3]:
seeds = list(range(42, 52))
particles, iterations = 30, 100
runs = []
for seed in seeds:
    beliefs, fitness, history = optimize(
        engine, train_active, df_train["Kerusakan"].to_numpy(), rules,
        initial, seed, particles, iterations,
    )
    result = metrics(y_test, engine.matrix(test_active, beliefs), classes)
    runs.append({"seed": seed, "beliefs": beliefs, "train_fitness": fitness,
                 "history": history, "metrics": result})
    print(f"Seed {seed}: fitness latih={fitness:.6f}, accuracy uji={result['accuracy']:.4%}, "
          f"macro F1 uji={result['macro_f1']:.4%}")

per_seed = pd.DataFrame([{
    "Seed": run["seed"], "Fitness latih": run["train_fitness"],
    "Accuracy": run["metrics"]["accuracy"], "Macro F1": run["metrics"]["macro_f1"],
    "Weighted F1": run["metrics"]["weighted_f1"],
    "Brier score": run["metrics"]["multiclass_brier"],
} for run in runs])
display(per_seed.style.format({"Fitness latih": "{:.6f}", "Accuracy": "{:.2%}",
                               "Macro F1": "{:.2%}", "Weighted F1": "{:.2%}",
                               "Brier score": "{:.4f}"}))

Seed 42: fitness latih=0.730348, accuracy uji=85.9459%, macro F1 uji=63.2608%
Seed 43: fitness latih=0.733137, accuracy uji=87.0270%, macro F1 uji=66.2370%
Seed 44: fitness latih=0.732280, accuracy uji=86.4865%, macro F1 uji=65.9707%
Seed 45: fitness latih=0.733125, accuracy uji=87.0270%, macro F1 uji=66.2370%
Seed 46: fitness latih=0.728713, accuracy uji=85.9459%, macro F1 uji=63.2608%
Seed 47: fitness latih=0.728838, accuracy uji=86.4865%, macro F1 uji=65.9202%
Seed 48: fitness latih=0.731806, accuracy uji=87.0270%, macro F1 uji=66.2818%
Seed 49: fitness latih=0.730948, accuracy uji=87.0270%, macro F1 uji=66.2370%
Seed 50: fitness latih=0.733190, accuracy uji=87.0270%, macro F1 uji=66.2370%
Seed 51: fitness latih=0.729930, accuracy uji=87.0270%, macro F1 uji=66.2370%


,Seed,Fitness latih,Accuracy,Macro F1,Weighted F1,Brier score
0,42,0.730348,85.95%,63.26%,85.47%,0.2989
1,43,0.733137,87.03%,66.24%,86.37%,0.2940
2,44,0.732280,86.49%,65.97%,86.04%,0.3002
3,45,0.733125,87.03%,66.24%,86.37%,0.2928
4,46,0.728713,85.95%,63.26%,85.47%,0.3002
5,47,0.728838,86.49%,65.92%,85.99%,0.2989
6,48,0.731806,87.03%,66.28%,86.41%,0.2926
7,49,0.730948,87.03%,66.24%,86.37%,0.2992
8,50,0.733190,87.03%,66.24%,86.37%,0.2921
9,51,0.729930,87.03%,66.24%,86.37%,0.2907


## 3. Rata-rata, seed terbaik

In [4]:
best = max(runs, key=lambda run: (run["metrics"]["macro_f1"], run["metrics"]["accuracy"]))
summary = per_seed[["Accuracy", "Macro F1", "Weighted F1", "Brier score"]].agg(["mean", "std"])
display(summary.style.format("{:.6f}"))
print(f"Seed terbaik: {best['seed']}; prediksi benar: "
      f"{round(best['metrics']['accuracy'] * len(y_test))}/{len(y_test)}")

mapping = [("Accuracy", "accuracy"), ("Macro F1", "macro_f1"),
           ("Weighted F1", "weighted_f1"), ("Brier score", "multiclass_brier")]
comparison = pd.DataFrame([{
    "Metrik": label,
    "DS awal": baseline[key],
    "DS PSO rata-rata": summary.loc["mean", label],
    "DS PSO terbaik": best["metrics"][key],
    "Perubahan terbaik dari DS": (baseline[key] - best["metrics"][key] if key == "multiclass_brier"
                                  else round(best["metrics"][key] * 100, 2) - round(baseline[key] * 100, 2)),
} for label, key in mapping]).set_index("Metrik")
comparison_display = comparison.astype(object).copy()
for label in comparison_display.index:
    for column in comparison_display.columns:
        value = comparison_display.loc[label, column]
        if label == "Brier score":
            comparison_display.loc[label, column] = f"{value:.4f}"
        elif column == "Perubahan terbaik dari DS":
            comparison_display.loc[label, column] = f"{value:+.2f} poin persentase"
        else:
            comparison_display.loc[label, column] = f"{value:.2%}"
display(comparison_display)
print("Perubahan Brier = Brier DS awal - Brier PSO terbaik (turun).")

,Accuracy,Macro F1,Weighted F1,Brier score
mean,0.867027,0.655880,0.861235,0.295963
std,0.004558,0.012327,0.003756,0.003812


Seed terbaik: 48; prediksi benar: 161/185


,DS awal,DS PSO rata-rata,DS PSO terbaik,Perubahan terbaik dari DS
Metrik,,,,
Accuracy,83.24%,86.70%,87.03%,+3.79 poin persentase
Macro F1,61.94%,65.59%,66.28%,+4.34 poin persentase
Weighted F1,83.10%,86.12%,86.41%,+3.31 poin persentase
Brier score,0.3922,0.2960,0.2926,0.0995


Perubahan Brier = Brier DS awal - Brier PSO terbaik (turun).
